Load excel with queries and included PMIDs

In [ ]:
import pandas as pd
import os
# Paths are relative to the repo root - run this notebook from there.


In [ ]:
# Load in data
reports = pd.read_excel('Overzicht_CI_rapporten_HealthBase.xlsx', sheet_name=1)

# Convert date column to date format usable in PubMed Query
reports['datum search '] = pd.to_datetime(reports['datum search '], errors='coerce').dt.strftime('%Y/%m/%d')
reports['Startdatum'] = pd.to_datetime(reports['Startdatum'], errors='coerce').dt.strftime('%Y/%m/%d')

# Forward fill the NaN values in the report number column
reports['CI-rapport nummer '] = reports['CI-rapport nummer '].ffill().astype(int)

# Remove PMID NA values
reports = reports.dropna(subset=['PMID geïncludeerde studies '])

# PMID ID's to string
reports['PMID geïncludeerde studies '] = reports['PMID geïncludeerde studies '].astype(str).str.strip()

# Remove reports with no PubMed hits according to the table
reports = reports[reports['PMID geïncludeerde studies '] != 'geen hits']

# Print number of unique reports
print('Number of articles: ', reports.shape[0])
print('Number of unique articles: ', reports['PMID geïncludeerde studies '].nunique())

# Print number of unique reports
print('Number of reports: ', reports['CI-rapport nummer '].nunique())

In [ ]:
reports

Use group-based split, so validation and final val are on query results unrelated to the train set. 

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Create a GroupShuffleSplit object to split into training and temporary test set 
gss_train_test = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=42)

# Using 'CI-rapport nummer' as the groups
groups = reports['CI-rapport nummer ']

# Perform the training and temporary test set split
train_idx, temp_test_idx = next(gss_train_test.split(reports, groups=groups))

# Create training and temporary test DataFrames
train_df = reports.iloc[train_idx]
temp_test_df = reports.iloc[temp_test_idx]

# Now, split the temporary test set into validation and test sets using GroupShuffleSplit with train_size=0.5
gss_val_test = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val_test.split(temp_test_df, groups=temp_test_df['CI-rapport nummer ']))

# Create validation and test DataFrames
val_df = temp_test_df.iloc[val_idx]
test_df = temp_test_df.iloc[test_idx]


In [ ]:
# remove duplicate pmids from splits
train_df = train_df.drop_duplicates(subset=['PMID geïncludeerde studies '], keep='first')
val_df = val_df.drop_duplicates(subset=['PMID geïncludeerde studies '], keep='first')
test_df = test_df.drop_duplicates(subset=['PMID geïncludeerde studies '], keep='first')

In [ ]:
# Store PMIDs in val set also present in train set (remove those articles from val set later)
exclude_pmids_val = [pmid for pmid in list(val_df['PMID geïncludeerde studies ']) if pmid in list(train_df['PMID geïncludeerde studies '])]

# Store PMIDs in test set also present in val and/or train set (remove those articles from test set later)
exclude_pmids_test = [pmid for pmid in list(test_df['PMID geïncludeerde studies ']) if (pmid in list(train_df['PMID geïncludeerde studies '])) or (pmid in list(val_df['PMID geïncludeerde studies ']))]

# Store PMIDS that should be available per set (if they are not found by the queries add manually)
pmids_train = list(train_df['PMID geïncludeerde studies '])
pmids_val = list(val_df['PMID geïncludeerde studies '][~val_df['PMID geïncludeerde studies '].isin(exclude_pmids_val)])
pmids_test = list(test_df['PMID geïncludeerde studies '][~test_df['PMID geïncludeerde studies '].isin(exclude_pmids_test)])

In [ ]:
# Check the number of unique PMIDs in the train, validation and test set
print(f"Inclusions train set: {len(pmids_train)}")
print(f"Inclusions validation set: {len(pmids_val)}")
print(f"Inclusions test set: {len(pmids_test)}")

# Get percentage of train, validation and test set
total_size = reports['PMID geïncludeerde studies '].nunique()

print(f"Percentage of train set: {len(pmids_train) / total_size}")
print(f"Percentage of validation set: {len(pmids_val) / total_size}")
print(f"Percentage of test set: {len(pmids_test) / total_size}")

Extend queries by date limits

In [ ]:
# Function to create the extended query
def extend_query(row):
    startdate = row['Startdatum']
    enddate = row['datum search ']
    query = row['Pubmed query ']
    if pd.isna(startdate):
        extended_query = f'({query}) AND ("1900/01/01"[Date - Publication]:"{enddate}"[Date - Publication])'
    else:
        extended_query = f'({query}) AND ("{startdate}"[Date - Publication]:"{enddate}"[Date - Publication])'

    return extended_query

# Apply the function to each row and create a new column 'extended_query'
train_df['extended_query'] = reports.apply(extend_query, axis=1)
val_df['extended_query'] = reports.apply(extend_query, axis=1)
test_df['extended_query'] = reports.apply(extend_query, axis=1)

# save as list of queries
queries_train = list(train_df['extended_query'][pd.notna(train_df['Pubmed query '])])
queries_val = list(val_df['extended_query'][pd.notna(val_df['Pubmed query '])])
queries_test = list(test_df['extended_query'][pd.notna(test_df['Pubmed query '])])
queries_train[0:5]

Use Query to input in PubMed and retrieve pmids, titles, authors, journal and abstracts. 

In [ ]:
from pymed import PubMed

# initiate pubmed instance and search
my_api_key = '8d6b8fa9bde1ec85f27cec49a16405f8ae09'
pubmed = PubMed(tool="ContraindicationTool", email="p.c.habets@lumc.nl")
pubmed.parameters.update({'api_key': my_api_key})


In [ ]:
# Define function to run all queries and return dataframe of results
def get_articles(queries_list: list, max_results=1000) -> pd.DataFrame:
    """Returns a DataFrame with articles found by all the queries in the queries_list"""

    # Initialize an empty list to store DataFrames of articles found by each query
    article_dfs = [] 

    for query in queries_list:
        assert isinstance(query, str), 'query must be a string'

        # call query
        results = pubmed.query(query, max_results=max_results)

        # appending results to list
        articleList = []
        articleInfo = []

        for article in results:
            articleDict = article.toDict()
            if 'isbn' not in articleDict.keys(): # Remove books: ISBN value only found for citations of books and book chapters, see: https://www.nlm.nih.gov/bsd/mms/medlineelements.html#pt
                articleList.append(articleDict)

        print(f'{len(articleList)} articles found')

        # extract info and turn list of dictionaries into dataframe

        for article in articleList:

            # Sometimes article['pubmed_id'] contains list separated with comma - take first pubmedId in that list - that's article pubmedId
            pmid = article['pubmed_id'].partition('\n')[0]

            # Sometimes no journal listed
            if 'journal' in article:
                journal = article['journal']
            else:
                journal = None

            # Sometime 'keywords' is no key, sometimes 'keywords' is a key but empty value
            if 'keywords' in article:
                keywords = ", ".join([str(k) for k in article['keywords'] if k is not None])
            else:
                keywords = None


            # join zipped initials and lastnames from dictionaries of authors
            initials = []
            for init in article['authors']:
                if init['initials']:
                    initials.append(init['initials'])
                else:
                    initials.append("")
            lastnames = []
            for name in article['authors']:
                if name['lastname']:
                    lastnames.append(name['lastname'])
                else:
                    lastnames.append("")
            authors = ", ".join([' '.join(z) for z in zip(initials, lastnames)])

            # append article info to dictionary
            articleInfo.append(
                {'pubmed_id': pmid,
                'title': article['title'],
                'publication_date': article['publication_date'],
                'journal': journal,
                'abstract': article['abstract'],
                'authors': authors,
                'keywords': keywords
                }
            )

        # Generate Pandas DataFrame from list of dictionaries
        articles_df = pd.DataFrame.from_dict(articleInfo)

        # # Remove trailing/leading whitespaces and explicit double quotations from Title and Abstract, journal, authors and keywords
        # clean = ['title', 'journal', 'abstract', 'authors', 'keywords']
        # for column in clean:
        #     articles_df[f'{column}'] = articles_df[f'{column}'].str.strip().str.replace('"', '')

        # Remove trailing/leading whitespaces and explicit double quotations from Title, Journal, Abstract, Authors, and Keywords
        clean = ['title', 'journal', 'abstract', 'authors', 'keywords']
        for column in clean:
            if column in articles_df.columns:  # Check if the column exists
                articles_df[column] = articles_df[column].str.strip().str.replace('"', '')


        # Append the DataFrame of the current query to a list
        article_dfs.append(articles_df)

    # Concatenate all the DataFrames in the list into one DataFrame
    final_df = pd.concat(article_dfs, ignore_index=True)

    return final_df


# Run Queries

### Train Articles

In [ ]:
results_train = [] # List to store results of each query

try_index = 0 # To keep track of the last successful index

try:
    for i in range(try_index, len(queries_train)):
        query = queries_train[i] # Get the query
        result = get_articles([query]) # Call your function with the query
        results_train.append(result) # Append the result to the list
        print(f'Successfully processed query at index {i}')
        try_index = i + 1 # Increment to the next index

except Exception as e:
    print(f'An error encountered: {e}')
    print(f'Stopping at query index {try_index}. Please restart from this point.')



In [ ]:
try_index

In [ ]:
import time

# After an error, continue from last_successful_index
try:
    for i in range(try_index, len(queries_train)):
        query = queries_train[i] # Get the query
        result = get_articles([query]) # Call your function with the query
        results_train.append(result) # Append the result to the list
        print(f'Successfully processed query at index {i}')
        try_index = i + 1 # Increment to the next index
        # time.sleep(3)
     
except Exception as e:
    print(f'An error encountered {e}')
    print(f'Stopping at query index {i}. Please restart from this point.')


In [ ]:
# concatenate all chunks into a single DataFrame
final_results_train = pd.concat(results_train, ignore_index=True)

In [ ]:
final_results_train

In [ ]:
# Get unique results from the results dataframe
unique_results_train = final_results_train.drop_duplicates(subset='pubmed_id', keep='first')
unique_results_train.nunique()

In [ ]:
# Add case column to unique_results_train
unique_results_train['case'] = unique_results_train['pubmed_id'].astype(str).isin(pmids_train).astype(int)
unique_results_train.shape

In [ ]:
unique_results_train['pubmed_id'].astype(str).isin(pmids_train).value_counts()

In [ ]:
# Get missing PMIDs (n = 331)]
missing_pmids_train = [pmid for pmid in pmids_train if pmid not in list(unique_results_train['pubmed_id'])]
print(f'{len(missing_pmids_train)} missing PMIDs in train set')

In [ ]:
# Use pubmeds to retrieve the articles (there is one pmid retrieving 0 articles)
missing_results_train = [] # List to store results of each query
try_index = 0 # To keep track of the last successful index

try:
    for i in range(try_index, len(missing_pmids_train)):
        query = missing_pmids_train[i] # Get the query
        result = get_articles([query]) # Call your function with the query
        missing_results_train.append(result) # Append the result to the list
        print(f'Successfully processed query at index {i}')
        try_index = i + 1 # Increment to the next index

except Exception as e:
    print(f'An error encountered: {e}')
    print(f'Stopping at query index {try_index}. Please restart from this point.')



In [ ]:
try_index

In [ ]:
import time

# After an error, continue from last_successful_index
try:
    for i in range(try_index, len(missing_pmids_train)):
        query = missing_pmids_train[i] # Get the query
        result = get_articles([query]) # Call your function with the query
        missing_results_train.append(result) # Append the result to the list
        print(f'Successfully processed query at index {i}')
        try_index = i + 1 # Increment to the next index
        # time.sleep(3)
     
except Exception as e:
    print(f'An error encountered {e}')
    print(f'Stopping at query index {i}. Please restart from this point.')


In [ ]:
unique_results_train

In [ ]:
# concatenate all chunks into a single DataFrame
missing_articles_train = pd.concat(missing_results_train, ignore_index=True)

# add 'case' column with trainue (1)
missing_articles_train['case'] = 1

# concatenate to one complete dataframe
complete_df_train = pd.concat([missing_articles_train,unique_results_train], ignore_index=True)

# Create additional column with title + abstract (if abstract exists, else title only)
complete_df_train['TiAb'] = complete_df_train.apply(lambda row: row['title'] if pd.isna(row['abstract']) else row['title'] + ' ' + row['abstract'], axis=1)

# Put 'case' column in front, TiAb column next
cols = ['case'] + ['TiAb'] + [col for col in complete_df_train.columns if col != 'case' and col != 'TiAb']
complete_df_train = complete_df_train[cols]

complete_df_train.head()

In [ ]:
print(f'shape: {complete_df_train.shape}')
print(f"\nvalue_counts: \n{complete_df_train['case'].value_counts()}")

In [ ]:
# directory for outputs
out = "scripts/pubmed_output"
os.makedirs(out, exist_ok=True)

# Write results to file
complete_df_train.to_csv(f'{out}/healthbase_articles_train_unbalanced_55K.csv', index=False)

### Validation Articles

In [ ]:
results_val = [] # List to store results of each query

try_index = 0 # To keep track of the last successful index

try:
    for i in range(try_index, len(queries_val)):
        query = queries_val[i] # Get the query
        result = get_articles([query]) # Call your function with the query
        results_val.append(result) # Append the result to the list
        print(f'Successfully processed query at index {i}')
        try_index = i + 1 # Increment to the next index

except Exception as e:
    print(f'An error encountered: {e}')
    print(f'Stopping at query index {try_index}. Please restart from this point.')


In [ ]:
try_index

In [ ]:
import time

# After an error, continue from last_successful_index
try:
    for i in range(try_index, len(queries_val)):
        query = queries_val[i] # Get the query
        result = get_articles([query]) # Call your function with the query
        results_val.append(result) # Append the result to the list
        print(f'Successfully processed query at index {i}')
        try_index = i + 1 # Increment to the next index
        time.sleep(3)
     
except Exception as e:
    print(f'An error encountered {e}')
    print(f'Stopping at query index {i}. Please restart from this point.')

In [ ]:
# concatenate all chunks into a single DataFrame
final_results_val = pd.concat(results_val, ignore_index=True)

In [ ]:
len(results_val) == len(queries_val)

In [ ]:
# Get unique results from the results dataframe
unique_results_val = final_results_val.drop_duplicates(subset='pubmed_id', keep='first')
unique_results_val.nunique()

In [ ]:
# Remove PMIDs that are also in train and/or val set
unique_results_val = unique_results_val[~unique_results_val['pubmed_id'].isin(exclude_pmids_val)]

# Add case column to unique_results_val
unique_results_val['case'] = unique_results_val['pubmed_id'].astype(str).isin(pmids_val).astype(int)
unique_results_val.head()

# Get missing PMIDs (n = 44)]
missing_pmids_val = [pmid for pmid in pmids_val if pmid not in list(unique_results_val['pubmed_id'])]

In [ ]:
unique_results_val['case'].value_counts()


In [ ]:
# Use pubmeds to retrieve the articles (there is one pmid retrieving 0 articles)
missing_articles_val = get_articles(missing_pmids_val) 

In [ ]:
# add 'case' column with value (1)
missing_articles_val['case'] = 1

# concatenate to one complete dataframe
complete_df_val = pd.concat([missing_articles_val,unique_results_val], ignore_index=True)

# Create additional column with title + abstract (if abstract exists, else title only)
complete_df_val['TiAb'] = complete_df_val.apply(lambda row: row['title'] if pd.isna(row['abstract']) else row['title'] + ' ' + row['abstract'], axis=1)

# Put 'case' column in front, TiAb column next
cols = ['case'] + ['TiAb'] + [col for col in complete_df_val.columns if col != 'case' and col != 'TiAb']
complete_df_val = complete_df_val[cols]

complete_df_val.head()

In [ ]:
print(complete_df_val.shape)
complete_df_val['case'].value_counts()

In [ ]:
# directory for outputs
out = "scripts/pubmed_output"
os.makedirs(out, exist_ok=True)

# Write results to file
complete_df_val.to_csv(f'{out}/healthbase_articles_val_unbalanced_6.5K.csv', index=False)

### Test Articles

In [ ]:
results_test = [] # List to store results of each query

try_index = 0 # To keep track of the last successful index

try:
    for i in range(try_index, len(queries_test)):
        query = queries_test[i] # Get the query
        result = get_articles([query]) # Call your function with the query
        results_test.append(result) # Append the result to the list
        print(f'Successfully processed query at index {i}')
        try_index = i + 1

except Exception as e:
    print(f'An error encountered: {e}')
    print(f'Stopping at query index {try_index}. Please restart from this point.')


In [ ]:
try_index

In [ ]:
import time

# After an error, continue from last_successful_index
try:
    for i in range(try_index, len(queries_test)):
        query = queries_test[i] # Get the query
        result = get_articles([query]) # Call your function with the query
        results_test.append(result) # Append the result to the list
        print(f'Successfully processed query at index {i}')
        try_index = i + 1 # Increment to the next index
        time.sleep(1)
     
except Exception as e:
    print(f'An error encountered {e}')
    print(f'Stopping at query index {i}. Please restart from this point.')

In [ ]:
# concatenate all results into a single DataFrame
final_results_test = pd.concat(results_test, ignore_index=True)

In [ ]:
len(results_test) == len(queries_test)

In [ ]:
# Get unique results from the results dataframe
unique_results_test = final_results_test.drop_duplicates(subset='pubmed_id', keep='first')
unique_results_test.nunique()

In [ ]:
# Remove PMIDs that are also in train and/or val set
unique_results_test = unique_results_test[~unique_results_test['pubmed_id'].isin(exclude_pmids_test)]

In [ ]:
# Add case column to unique_results_test
unique_results_test['case'] = unique_results_test['pubmed_id'].astype(str).isin(pmids_test).astype(int)
unique_results_test.head()

In [ ]:
unique_results_test['case'].value_counts()

In [ ]:
# Get missing PMIDs (n = 53)]
missing_pmids_test = [pmid for pmid in pmids_test if pmid not in list(unique_results_test['pubmed_id'])]

In [ ]:
# Use pubmeds to retrieve the articles (there is one pmid retrieving 0 articles)
missing_articles_test = get_articles(missing_pmids_test) 

In [ ]:
# add 'case' column with value (1)
missing_articles_test['case'] = 1
missing_articles_test.head()

In [ ]:
# concatenate to one complete dataframe
complete_df_test = pd.concat([missing_articles_test,unique_results_test], ignore_index=True)

# Create additional column with title + abstract (if abstract exists, else title only)
complete_df_test['TiAb'] = complete_df_test.apply(lambda row: row['title'] if pd.isna(row['abstract']) else row['title'] + ' ' + row['abstract'], axis=1)

# Put 'case' column in front, TiAb column next
cols = ['case'] + ['TiAb'] + [col for col in complete_df_test.columns if col != 'case' and col != 'TiAb']
complete_df_test = complete_df_test[cols]

In [ ]:
print(complete_df_test.shape)
complete_df_test['case'].value_counts()

In [ ]:
# directory for outputs
out = "scripts/pubmed_output"
os.makedirs(out, exist_ok=True)

# Write results to file
complete_df_test.to_csv(f'{out}/healthbase_articles_test_unbalanced_4.5K.csv', index=False)

# Create balanced dataframes for model training/validation/testing 
#### Choose random controls matched by number of missing abstracts

In [ ]:
complete_df_train
complete_df_val
complete_df_test

In [ ]:
def get_balanced_set(complete_df: pd.DataFrame) -> pd.DataFrame:
    """Function to return a balanced set of cases and controls matched by percentage of title-only"""

    # Step 1: count number of cases without abstract
    n_no_ab = pd.isna(complete_df[complete_df['case'] == 1]['abstract']).sum()
    n_with_ab = complete_df[complete_df['case'] == 1].shape[0] - n_no_ab

    # Step 2: split in cases and control (with and without) df
    df_cases = complete_df[complete_df['case'] == 1]
    df_controls_noAb = complete_df[(complete_df['case'] == 0) & (complete_df['abstract'].isna())]
    df_controls_TiAb = complete_df[(complete_df['case'] == 0) & (complete_df['abstract'].notna())]

    # Step 3: randomly draw a matching set of controls for each case (with and without abstract)
    controls_sample_noAb = df_controls_noAb.sample(n=n_no_ab, random_state=101)
    controls_sample_TiAb = df_controls_TiAb.sample(n=n_with_ab, random_state=101)

    # Step 4: concatenate the two DataFrames to get the balanced dataset
    balanced_df = pd.concat([df_cases, controls_sample_noAb, controls_sample_TiAb], axis=0).reset_index(drop=True)

    # Step 5: shuffle the balanced DataFrame
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    return balanced_df

    

In [ ]:
balanced_df_train = get_balanced_set(complete_df_train)
balanced_df_val = get_balanced_set(complete_df_val)
balanced_df_test = get_balanced_set(complete_df_test)

In [ ]:
for set in [balanced_df_train, balanced_df_val, balanced_df_test]:
    print(set.shape)
    print(sum(set['case']==1))

Save balanced dataframes for later use

In [ ]:
# directory for outputs
out = "scripts/pubmed_output"
os.makedirs(out, exist_ok=True)

balanced_df_train.to_csv(f'{out}/healthbase_articles_balanced_train.csv', index=False)
balanced_df_val.to_csv(f'{out}/healthbase_articles_balanced_val.csv', index=False)
balanced_df_test.to_csv(f'{out}/healthbase_articles_balanced_test.csv', index=False)